In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
# Load dataset (single location)
df = pd.read_csv('walmart-sales-dataset-of-45stores.csv', parse_dates=['Date'], dayfirst=True)

In [4]:
# Baseline reference (before preprocessing and tuning)
# A quick initial run achieved a low test R² around 0.16.
# We record that here as a baseline for comparison with the improved pipeline below.
baseline_r2_pre_preprocessing = 0.16
print(f"Baseline test R² (before preprocessing and tuning): {baseline_r2_pre_preprocessing:.2f}")

Baseline test R² (before preprocessing and tuning): 0.16


In [5]:
# Handle missing values and basic checks
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns
df[num_cols] = SimpleImputer(strategy='mean').fit_transform(df[num_cols])
df[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df[cat_cols])
print('Loaded rows, cols:', df.shape)
print(df.dtypes)

ValueError: SimpleImputer does not support data with dtype datetime64[ns]. Please provide either a numeric array (with a floating point or integer dtype) or categorical data represented either as an array with integer dtype or an array of string values with an object dtype.

In [ ]:
# Feature engineering: time features & grouped lags/rolling
df = df.sort_values(['Date']).reset_index(drop=True) if 'Date' in df.columns else df
if 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfWeek'] = df['Date'].dt.dayofweek
# Create group-based lags if Store/Dept present, else global lags
group_cols = [c for c in ['Store','Dept'] if c in df.columns]
if group_cols:
    df = df.sort_values(group_cols + (['Date'] if 'Date' in df.columns else [])).reset_index(drop=True)
    df['Sales_Lag1'] = df.groupby(group_cols)['Weekly_Sales'].shift(1)
    df['Sales_Roll4'] = df.groupby(group_cols)['Weekly_Sales'].apply(lambda x: x.shift(1).rolling(4).mean())
    df['store_dept_mean'] = df.groupby(group_cols)['Weekly_Sales'].transform('mean')
else:
    df['Sales_Lag1'] = df['Weekly_Sales'].shift(1)
    df['Sales_Roll4'] = df['Weekly_Sales'].shift(1).rolling(4).mean()
# Drop rows with NaNs introduced by lags/rolling
df = df.dropna().reset_index(drop=True)
# Drop Date column after extracting features to avoid scaling issues
if 'Date' in df.columns:
    df = df.drop(columns=['Date'])

TypeError: incompatible index of inserted column with frame index

In [ ]:
# Frequency-encode high-cardinality categoricals (Store/Dept) and drop original columns
for c in ['Store', 'Dept']:
    if c in df.columns:
        freq = df[c].value_counts(normalize=True)
        df[c + '_freq'] = df[c].map(freq).fillna(0)
        df = df.drop(columns=[c])

In [ ]:
# Define X and y, and a time-aware split (first 80% train, last 20% test)
y = df['Weekly_Sales']
X = df.drop(columns=['Weekly_Sales'])
cut = int(len(df) * 0.8)
X_train = X.iloc[:cut].copy()
X_test = X.iloc[cut:].copy()
y_train = y.iloc[:cut].copy()
y_test = y.iloc[cut:].copy()
print('Train rows, Test rows:', X_train.shape[0], X_test.shape[0])

Train rows, Test rows: 5148 1287


In [ ]:
# Minimal preprocessing pipeline + RidgeCV (time-aware CV)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
low_card = [c for c in cat_cols if X_train[c].nunique() <= 10]
preproc = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card),
], remainder='passthrough')
alphas = np.logspace(-3, 3, 30)
tscv = TimeSeriesSplit(n_splits=5)
pipe = Pipeline([('preproc', preproc), ('ridgecv', RidgeCV(alphas=alphas, cv=tscv, scoring='r2'))])
cv_scores = cross_val_score(pipe, X_train, y_train, cv=tscv, scoring='r2', n_jobs=-1)
print('TimeSeries CV R2 scores:', cv_scores)
print('CV mean R2: %.4f' % np.mean(cv_scores))
pipe.fit(X_train, y_train)
y_pred_test = pipe.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)
print('Test R2: %.4f | RMSE: %.2f | MAE: %.2f' % (r2_test, rmse_test, mae_test))
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_test, alpha=0.6, edgecolors='k')
mn, mx = min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('Actual Weekly_Sales')
plt.ylabel('Predicted Weekly_Sales')
plt.title('Parity plot: RidgeCV pipeline')
plt.grid(True)
plt.show()

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/pipeline.py", line 588, in _fit
    X, fitted_transformer = fit_transform_on

In [ ]:
# Baseline check: naive predictor = previous week's sales (if available)
if 'Sales_Lag1' in X_test.columns:
    baseline_pred = X_test['Sales_Lag1']
    print('Baseline (lag1) R2:', r2_score(y_test, baseline_pred))